
## 1 - 🔧 Preparação do Ambiente e Instalação de Bibliotecas

Nesta etapa, são instaladas as bibliotecas necessárias para execução do projeto, incluindo ferramentas de inteligência artificial, processamento de linguagem natural e síntese de voz.

In [ ]:
!pip install openai gtts

## 2 - 🔐 Configuração da Chave de API

Nesta etapa, realizamos a autenticação com a API da OpenAI, permitindo o uso dos modelos de inteligência artificial.

In [ ]:
import os

openai.api_key = os.getenv("OPENAI_API_KEY")

## 3 - 🎤 Entrada de Áudio

Nesta etapa, o usuário pode escolher entre enviar um arquivo de áudio ou gravar um novo áudio diretamente pelo navegador.

Essa flexibilidade permite diferentes formas de interação com o sistema, tornando o uso mais acessível e intuitivo.

In [ ]:
modo = input("Escolha o tipo de entrada:\n1 - Upload de áudio\n2 - Gravar áudio\nDigite a opção: ")

# =========================
# OPÇÃO 1 - UPLOAD DE ÁUDIO
# =========================
if modo == "1":
    from google.colab import files

    print("📂 Envie um arquivo de áudio (.wav, .mp3 ou .m4a):")
    uploaded = files.upload()

    audio_file = list(uploaded.keys())[0]
    tipo_entrada = "audio"

    print("✅ Áudio recebido:", audio_file)


# =========================
# OPÇÃO 2 - GRAVAÇÃO
# =========================
elif modo == "2":
    from IPython.display import Javascript, display
    from google.colab import output
    import base64

    def record_audio(filename="audio_gravado.wav", duration=5):
        display(Javascript(f"""
        async function recordAudio() {{
          const stream = await navigator.mediaDevices.getUserMedia({{ audio: true }});
          const recorder = new MediaRecorder(stream);
          let chunks = [];

          recorder.ondataavailable = e => chunks.push(e.data);

          recorder.start();

          await new Promise(resolve => setTimeout(resolve, {duration * 1000}));

          recorder.stop();

          return new Promise(resolve => {{
            recorder.onstop = () => {{
              const blob = new Blob(chunks);
              const reader = new FileReader();
              reader.readAsDataURL(blob);
              reader.onloadend = () => resolve(reader.result);
            }};
          }});
        }}
        recordAudio();
        """))

        data = output.eval_js("recordAudio()")
        audio_data = base64.b64decode(data.split(',')[1])

        with open(filename, 'wb') as f:
            f.write(audio_data)

        return filename

    audio_file = record_audio()
    tipo_entrada = "audio"

    print("🎙️ Áudio gravado com sucesso!")


else:
    print("❌ Opção inválida")

Escolha o tipo de entrada:
1 - Upload de áudio
2 - Gravar áudio
Digite a opção: 1
📂 Envie um arquivo de áudio (.wav, .mp3 ou .m4a):


Saving Pergunta teste-VirtualCare.m4a to Pergunta teste-VirtualCare (10).m4a
✅ Áudio recebido: Pergunta teste-VirtualCare (10).m4a


## 4 - 🗣️ Conversão de Voz para Texto

Utilizando o modelo Whisper, o áudio enviado é convertido em texto para posterior análise.

In [ ]:
audio_file = list(uploaded.keys())[0]

with open(audio_file, "rb") as f:
    transcript = openai.audio.transcriptions.create(
        model="whisper-1",
        file=f,
        language="pt"
    )

texto = transcript.text
print("🗣️ Você disse:", texto)

🗣️ Você disse: Chat, o pediatra passou o clavurinho para o meu filho. Gostaria de saber se preciso dar para ele com comida ou com estômago vazio.


## 5 - 🤖 Geração de Resposta com Inteligência Artificial

O texto transcrito é enviado ao modelo de linguagem, que gera uma resposta inteligente baseada na pergunta do usuário.

In [ ]:
response = openai.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Responda sempre em português."},
        {"role": "user", "content": texto}
    ]
)

resposta = response.choices[0].message.content

# 👇 COLOCA AQUI
print("Texto enviado:", texto)
print("Resposta da IA:", resposta)

Texto enviado: Chat, o pediatra passou o clavurinho para o meu filho. Gostaria de saber se preciso dar para ele com comida ou com estômago vazio.
Resposta da IA: O Clavurinho (que contém amoxicilina e ácido clavulânico) geralmente pode ser administrado com ou sem alimentos. Porém, dar o medicamento com alimentos pode ajudar a reduzir a possibilidade de desconforto gástrico. É sempre bom seguir as orientações específicas do pediatra que acompanha seu filho. Se tiver dúvidas, não hesite em consultar o médico novamente.


## 6 - 🔊 Conversão de Texto para Voz

Nesta etapa, a resposta gerada pela inteligência artificial é convertida em áudio utilizando a biblioteca gTTS.

Antes da conversão, o sistema valida se há conteúdo na resposta, evitando a geração de áudios vazios.

In [ ]:
from gtts import gTTS
from IPython.display import Audio, display

# Verificação de segurança
if resposta is None or resposta.strip() == "":
    print("⚠️ Não foi possível gerar áudio: resposta vazia.")
else:
    print("🧠 Resposta gerada:")
    print(resposta)

    tts = gTTS(text=resposta, lang="pt")
    tts.save("resposta.mp3")

    print("✅ Áudio gerado com sucesso!")

🧠 Resposta gerada:
O Clavurinho (que contém amoxicilina e ácido clavulânico) geralmente pode ser administrado com ou sem alimentos. Porém, dar o medicamento com alimentos pode ajudar a reduzir a possibilidade de desconforto gástrico. É sempre bom seguir as orientações específicas do pediatra que acompanha seu filho. Se tiver dúvidas, não hesite em consultar o médico novamente.
✅ Áudio gerado com sucesso!


## 7 - 🎧 Reprodução do Áudio Gerado

Nesta etapa, o áudio gerado é reproduzido para o usuário, permitindo a interação completa por voz.

In [ ]:
from IPython.display import Audio, display
import os

# Verificação antes de reproduzir
if os.path.exists("resposta.mp3"):
    print("🔊 Reproduzindo áudio...")
    display(Audio("resposta.mp3"))
else:
    print("❌ Arquivo de áudio não encontrado.")

🔊 Reproduzindo áudio...


## 8 - 📊 Fluxo do Sistema

O funcionamento do sistema segue as seguintes etapas:

1. Seleção do tipo de entrada (upload de áudio ou gravação)
2. Captura do áudio do usuário
3. Pré-processamento do áudio (conversão para formato WAV, quando necessário)
4. Transcrição do áudio para texto (Speech-to-Text)
5. Processamento do texto com Inteligência Artificial
6. Geração da resposta em linguagem natural
7. Conversão da resposta em áudio (Text-to-Speech)
8. Reprodução da resposta em áudio para o usuário